In [1]:
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [ ]:
# first things first, extract tables from database

In [2]:
# assuming we extract all 4 databases (MASTER, avg_df, known_val_feature_df, seasonal_constraints), preprocess master DB, THENNN (also preprocessing may be done backend, or maybe not)
master_df_p1 = master_df.drop(columns=["p2_outbreak_value", "p3_outbreak_value"])
master_df_p2 = master_df.drop(columns=["p1_outbreak_value", "p3_outbreak_value"])
master_df_p3 = master_df.drop(columns=["p1_outbreak_value", "p2_outbreak_value"])


NameError: name 'master_df' is not defined

In [ ]:
# Nowwwww, we gotta put each of those into SARIMAX
# first, we put ONE of them into SARIMAX. Lets do it slowllyyy..........
# lets first do master_df_p1
p1_y = master_df_p1["p1_outbreak_value"]
p1_x = master_df_p1[[col1, col2, col3, ...]] # everything except p1 outbreak value (ALSO, EXCEPT THE DATE. SAME FOR P2,P3)
p1_model = SARIMAX(
    p1_y,
    exog=p1_x,
    order=(1,0,1), # these params may be changed
    seasonal_order=(1,1,1,12) # params may change
)
p1_result = p1_model.fit()

# now master_df_p2
p2_y = master_df_p2["p2_outbreak_value"]
p2_x = master_df_p2[[col1, col2, col3, ...]] # everything except p2 outbreak value
p2_model = SARIMAX(
    p2_y,
    exog=p2_x,
    order=(1,0,1), # these params may be changed
    seasonal_order=(1,1,1,12) # params may change
)
p2_result = p2_model.fit()

# master_df_p3
p3_y = master_df_p3["p3_outbreak_value"]
p3_x = master_df_p3[[col1, col2, col3, ...]] # everything except p3 outbreak value
p3_model = SARIMAX(
    p3_y,
    exog=p3_x,
    order=(1,0,1), # these params may be changed
    seasonal_order=(1,1,1,12) # params may change
)
p3_result = p3_model.fit()

In [30]:
# now we should make the apply dataset (same for all p1, p2, p3)

# make sure date column in og DF is in datetime format
date_col = "Date"   # change this to your actual date column name

master_without_outputs = master_df_p1.drop(columns=["p1_outbreak_value"])

# get the latest date in the original df
last_date = master_without_outputs[date_col].max()

# first day of next 3 months
future_dates = pd.date_range(
    start=(last_date + pd.offsets.MonthBegin(1)).normalize(),
    periods=3,
    freq="MS"
)

# create empty df with same columns, 3 rows
future_df = pd.DataFrame(columns=master_without_outputs.columns, index=range(3)) # this is our apply df

# fill date column
future_df[date_col] = future_dates


NameError: name 'master_df_p1' is not defined

In [17]:
# now we should fill up the apply dataset with all em values...
# get month number for each future row
future_df["MonthNum"] = future_df[date_col].dt.month

# fill chosen features from averages table
for i, row in future_df.iterrows():
    month_num = row["MonthNum"]

    # get the row from avg_df for that month
    avg_row = avg_df[avg_df["Month"] == month_num] # here avg_df is extracted from our master storage 

    if not avg_row.empty:
        for col in avg_features:
            if col in future_df.columns and col in avg_df.columns:
                future_df.at[i, col] = avg_row.iloc[0][col]

# fill features from known_value_features table
for i, row in future_df.iterrows():
    month_num = row["Month"]
    known_row = known_val_feature_df[known_val_feature_df["Month"] == month_num] # here known_val_feature_df is also extracted from master storage

    if not known_row.empty:
        for col in known_val_feature_df.columns:
            if col != "Month" and col in future_df.columns:
                future_df.at[i, col] = known_row.iloc[0][col]

# remove helper column
future_df = future_df.drop(columns="MonthNum")


In [19]:
# Now we gotta run these through each of the models for p1, p2, p3 lollllll
# ok so firstly, the format of future_df must be THE SAME as our training df (preprocessed master DB), mneaning that
# format and dtypes of each column shld be same, etc.... also make sure theres no NaN values or any of that

In [ ]:
future_X = future_df[[col1, col2, col3, ...]] # all columns except DATE (theres no output column here anyway)
forecast_p1 = p1_result.forecast(steps=len(future_X), exog=future_X)
forecast_p2 = p2_result.forecast(steps=len(future_X), exog=future_X)
forecast_p3 = p3_result.forecast(steps=len(future_X), exog=future_X)